In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer


URLS = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Data_mining",
    "https://en.wikipedia.org/wiki/Business_intelligence",
]

STOPWORDS = set("""
a an the is are was were be been being of to in on for and or but with as at by
from this that these those it its it's not no yes you your we our they their he
she his her him them i my me us if then so than too very can could will would
just about into over under more most also up down out than there here what when
where who whom which why how all any each other some such only own same do does
did having have has had
""".split())


def scrape_page(url: str) -> str:
    """Fetch a page and return its cleaned visible text."""
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"[!] Failed to fetch {url}: {e}")
        return ""

    soup = BeautifulSoup(resp.text, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
        tag.decompose()

    container = soup.find("article") or soup.find("main") or soup
    paragraphs = container.find_all("p")
    text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)

    return text.strip()


def clean_and_tokenize(text: str):
    text = text.lower()
    words = re.findall(r"[a-z]{3,}", text)
    return [w for w in words if w not in STOPWORDS]


def main():
    docs = {}
    print("Scraping pages...\n")
    for url in URLS:
        text = scrape_page(url)
        if text:
            docs[url] = text
            print(f"  [ok] {url} ({len(text.split())} words)")
        else:
            print(f"  [skip] {url} (no content)")

    if not docs:
        print("No content scraped. Check your URLs / internet access.")
        return

    all_words = []
    for text in docs.values():
        all_words.extend(clean_and_tokenize(text))

    print("\n=== Overall Trending Words (across all pages) ===")
    for word, count in Counter(all_words).most_common(15):
        print(f"  {word:<15} {count}")

    urls = list(docs.keys())
    corpus = [docs[u] for u in urls]

    vectorizer = TfidfVectorizer(
        stop_words=list(STOPWORDS), max_features=2000, token_pattern=r"[a-zA-Z]{3,}"
    )
    tfidf_matrix = vectorizer.fit_transform(corpus).toarray()  # type: ignore[attr-defined]  # spmatrix.toarray() exists at runtime; stub gap
    terms = list(vectorizer.get_feature_names_out())

    print("\n=== Top Topics Per Page (TF-IDF) ===")
    for i, url in enumerate(urls):
        row = tfidf_matrix[i]
        top_indices = row.argsort()[::-1][:8]
        top_terms = [str(terms[j]) for j in top_indices if row[j] > 0]
        print(f"\n  {url}")
        print(f"    Topics: {', '.join(top_terms)}")


if __name__ == "__main__":
    main()

Scraping pages...

  [ok] https://en.wikipedia.org/wiki/Artificial_intelligence (14656 words)
  [ok] https://en.wikipedia.org/wiki/Data_mining (2879 words)
  [ok] https://en.wikipedia.org/wiki/Business_intelligence (1741 words)

=== Overall Trending Words (across all pages) ===
  data            224
  intelligence    101
  used            72
  mining          72
  learning        67
  artificial      53
  machine         53
  business        51
  use             48
  may             47
  information     46
  human           44
  models          41
  many            40
  research        37

=== Top Topics Per Page (TF-IDF) ===

  https://en.wikipedia.org/wiki/Artificial_intelligence
    Topics: human, intelligence, learning, data, used, artificial, machine, reasoning

  https://en.wikipedia.org/wiki/Data_mining
    Topics: data, mining, patterns, information, database, copyright, used, set

  https://en.wikipedia.org/wiki/Business_intelligence
    Topics: data, business, intelligence, a